# Cuaderno 2 — Construcción de features y variable objetivo

**Tesis:** Predicción del Punto de Equilibrio de Flujo de Caja en Empresas de Crecimiento del Sector Tecnológico

**Universidad EAFIT · 2026**

---

## Objetivo

Transformar los datos crudos del universo tecnológico (Cuaderno 1) en un
dataset listo para el modelo de machine learning: destrimestralización,
transformación a formato panel, construcción de la variable objetivo Y,
e ingeniería de 11 variables derivadas.

---

## Decisiones metodológicas

**Variable objetivo:** Y=1 si el FCF operativo promedio de los próximos 4
trimestres es positivo, Y=0 si no. Se usa el promedio (no el máximo) para
capturar breakeven sostenido, no un trimestre positivo puntual.

**Filtro de universo:** solo trimestres con FCF operativo negativo — la
pregunta de investigación aplica únicamente a empresas que hoy queman caja.

**Outliers:** winsorización al percentil 1-99, preservando la información
extrema sin que distorsione el entrenamiento.

**Cobertura mínima:** se eliminan variables con menos del 15% de datos
disponibles, recalculado específicamente sobre el universo tecnológico
(no se asume que las mismas variables del modelo general apliquen aquí).

## 1. Carga y limpieza de fechas

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Carga optimizada ---
df = pd.read_csv(
    "../datos/crudos/variables_financieras_tech.csv.gz",
    compression="gzip",
    usecols=["ticker", "sector", "bolsa", "variable", "form",
             "start", "end", "val", "fp", "fy"],
    dtype={
        "ticker": "category", "variable": "category", "form": "category",
        "bolsa": "category", "sector": "category", "fp": "category",
        "val": "float32",
    }
)
df["start"] = pd.to_datetime(df["start"], errors="coerce")
df["end"]   = pd.to_datetime(df["end"],   errors="coerce")
df["fy"]    = pd.to_numeric(df["fy"], errors="coerce").astype("Int32")

print(f"Dataset cargado: {df.shape}")
print(f"Empresas: {df['ticker'].nunique():,}")

# --- Limpieza de fechas imposibles ---
n_antes = len(df)
df = df[(df["end"].dt.year >= 2000) & (df["end"].dt.year <= 2026)].copy()
print(f"Registros eliminados por fecha inválida: {n_antes - len(df):,}")

# --- Separar variables de balance vs. flujo ---
variables_balance = [
    "activos_corrientes", "activos_totales", "caja",
    "cuentas_por_cobrar", "cuentas_por_pagar", "deuda_corto_plazo",
    "deuda_largo_plazo", "inventario", "pasivos_corrientes",
    "activos_intangibles", "goodwill", "ppe_neto", "patrimonio",
    "retained_earnings", "ingresos_diferidos", "impuestos_pagados",
    "acciones_circulacion"
]
df_balance = df[df["variable"].isin(variables_balance)].copy()
df_flujo   = df[~df["variable"].isin(variables_balance)].copy()
df_balance = df_balance[df_balance["fp"].isin(["Q1", "Q2", "Q3", "Q4"])].copy()

# --- Destrimestralización de variables de flujo ---
df_flujo["duracion_dias"] = (df_flujo["end"] - df_flujo["start"]).dt.days
df_flujo_puro = df_flujo[df_flujo["duracion_dias"].between(80, 100)].copy()
df_flujo_acum = df_flujo[~df_flujo["duracion_dias"].between(80, 100)].copy()
df_flujo_acum = df_flujo_acum[df_flujo_acum["duracion_dias"].between(150, 400)].copy()

df_flujo_acum = df_flujo_acum.sort_values(["ticker", "variable", "fy", "end"]).copy()
df_flujo_acum["val_anterior"] = df_flujo_acum.groupby(
    ["ticker", "variable", "fy"], observed=True
)["val"].shift(1)
df_flujo_acum["val_trimestral"] = df_flujo_acum["val"] - df_flujo_acum["val_anterior"]
mask_q1 = df_flujo_acum["val_anterior"].isna()
df_flujo_acum.loc[mask_q1, "val_trimestral"] = df_flujo_acum.loc[mask_q1, "val"]
df_flujo_acum = df_flujo_acum.dropna(subset=["val_trimestral"]).copy()
df_flujo_acum["val"] = df_flujo_acum["val_trimestral"].astype("float32")
df_flujo_acum = df_flujo_acum.drop(columns=["val_anterior", "val_trimestral", "duracion_dias"])

df_flujo_final = pd.concat([
    df_flujo_puro.drop(columns=["duracion_dias"]), df_flujo_acum
], ignore_index=True)

print(f"\nFlujo total final: {len(df_flujo_final):,}")
print(f"Balance final: {len(df_balance):,}")

# --- Consolidar y pivot a formato ancho ---
df_limpio = pd.concat([df_flujo_final, df_balance], ignore_index=True)

df_ancho = df_limpio.pivot_table(
    index=["ticker", "sector", "bolsa", "end"],
    columns="variable", values="val", aggfunc="mean", observed=True
).reset_index()
df_ancho.columns.name = None
df_ancho = df_ancho.sort_values(["ticker", "end"]).reset_index(drop=True)

print(f"\nShape después del pivot: {df_ancho.shape}")
print(f"Empresas: {df_ancho['ticker'].nunique():,}")
print(f"Trimestres por empresa (promedio): {len(df_ancho) / df_ancho['ticker'].nunique():.1f}")

Dataset cargado: (640523, 10)
Empresas: 256
Registros eliminados por fecha inválida: 0

Flujo total final: 386,191
Balance final: 187,336

Shape después del pivot: (10988, 44)
Empresas: 256
Trimestres por empresa (promedio): 42.9


In [2]:
# --- Construcción de la variable objetivo Y ---
def calcular_fcf_futuro(grupo):
    """Para cada trimestre T, promedia el FCF en T+1, T+2, T+3, T+4."""
    fcf = grupo["fcf_operativo"].values
    resultado = []
    for i in range(len(fcf)):
        futuros = fcf[i+1:i+5]
        if len(futuros) == 4 and not np.isnan(futuros).all():
            resultado.append(np.nanmean(futuros))
        else:
            resultado.append(np.nan)
    grupo = grupo.copy()
    grupo["fcf_promedio_futuro"] = resultado
    return grupo

df_ancho = df_ancho.groupby("ticker", group_keys=False, observed=True).apply(
    calcular_fcf_futuro, include_groups=True
)

df_ancho["Y"] = (df_ancho["fcf_promedio_futuro"] > 0).astype(float)
df_ancho.loc[df_ancho["fcf_promedio_futuro"].isna(), "Y"] = np.nan

# --- Filtrar universo: solo empresas con FCF negativo hoy, y Y válida ---
df_modelo = df_ancho[
    (df_ancho["fcf_operativo"] < 0) &
    (df_ancho["Y"].notna())
].copy()

print(f"Total filas (antes de filtro): {len(df_ancho):,}")
print(f"Filas con Y válida: {df_ancho['Y'].notna().sum():,}")
print(f"Dataset del modelo (FCF negativo + Y válida): {len(df_modelo):,}")
print(f"Empresas únicas en dataset final: {df_modelo['ticker'].nunique():,}")
print(f"Y=1 (breakeven): {(df_modelo['Y']==1).sum():,} ({df_modelo['Y'].mean()*100:.1f}%)")
print(f"Y=0 (no alcanza): {(df_modelo['Y']==0).sum():,} ({(1-df_modelo['Y'].mean())*100:.1f}%)")

df_modelo["ano"] = pd.to_datetime(df_modelo["end"]).dt.year
print(f"\nDistribución por año:")
print(df_modelo["ano"].value_counts().sort_index())

Total filas (antes de filtro): 10,988
Filas con Y válida: 9,466
Dataset del modelo (FCF negativo + Y válida): 3,390
Empresas únicas en dataset final: 255
Y=1 (breakeven): 1,923 (56.7%)
Y=0 (no alcanza): 1,467 (43.3%)

Distribución por año:
ano
2008      2
2009     27
2010     88
2011    108
2012    121
2013    131
2014     96
2015    115
2016    116
2017    191
2018    215
2019    283
2020    316
2021    386
2022    457
2023    380
2024    352
2025      6
Name: count, dtype: int64


/tmp/ipykernel_7261/917637006.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_ancho = df_ancho.groupby("ticker", group_keys=False, observed=True).apply(


In [3]:
# --- Ingeniería de variables derivadas ---
df_modelo = df_modelo.copy()

df_modelo["burn_rate"]        = df_modelo["fcf_operativo"].abs()
df_modelo["runway"]           = df_modelo["caja"] / df_modelo["burn_rate"]
df_modelo["capital_trabajo"]  = df_modelo["activos_corrientes"] - df_modelo["pasivos_corrientes"]
df_modelo["margen_bruto"]     = df_modelo["utilidad_bruta"] / df_modelo["ingresos"]
df_modelo["margen_operativo"] = df_modelo["utilidad_operativa"] / df_modelo["ingresos"]
df_modelo["margen_neto"]      = df_modelo["utilidad_neta"] / df_modelo["ingresos"]
df_modelo["ratio_id"]         = df_modelo["gastos_id"] / df_modelo["ingresos"]
df_modelo["ratio_deuda"]      = df_modelo["deuda_largo_plazo"] / df_modelo["activos_totales"]
df_modelo["ratio_corriente"]  = df_modelo["activos_corrientes"] / df_modelo["pasivos_corrientes"]
df_modelo["rotacion_activos"] = df_modelo["ingresos"] / df_modelo["activos_totales"]
df_modelo["intensidad_capex"] = df_modelo["capex"].abs() / df_modelo["ingresos"]

# --- Winsorización al percentil 1-99 ---
variables_winsorizables = [
    "runway", "burn_rate", "capital_trabajo",
    "margen_bruto", "margen_operativo", "margen_neto",
    "ratio_id", "ratio_deuda", "ratio_corriente",
    "rotacion_activos", "intensidad_capex"
]

for var in variables_winsorizables:
    p01 = df_modelo[var].quantile(0.01)
    p99 = df_modelo[var].quantile(0.99)
    df_modelo[var] = df_modelo[var].clip(lower=p01, upper=p99)

print("Variables derivadas creadas y winsorizadas.")
print(f"Shape: {df_modelo.shape}")

# --- Control de calidad: cobertura de datos por variable ---
no_features = ["ticker", "sector", "bolsa", "end", "ano", "fcf_promedio_futuro", "Y"]
features = [c for c in df_modelo.columns if c not in no_features]

cobertura = df_modelo[features].notna().mean().sort_values()
print(f"\nVariables con cobertura < 15%:")
print(cobertura[cobertura < 0.15])

cols_eliminar = cobertura[cobertura < 0.15].index.tolist()
df_modelo = df_modelo.drop(columns=cols_eliminar, errors="ignore")
features = [f for f in features if f not in cols_eliminar]

print(f"\nVariables eliminadas por baja cobertura: {cols_eliminar}")
print(f"Features finales: {len(features)}")
print(f"Shape final: {df_modelo.shape}")

Variables derivadas creadas y winsorizadas.
Shape: (3390, 58)

Variables con cobertura < 15%:
deuda_corto_plazo    0.002950
intereses_pagados    0.137758
dtype: float64

Variables eliminadas por baja cobertura: ['deuda_corto_plazo', 'intereses_pagados']
Features finales: 49
Shape final: (3390, 56)


/home/vscode/.local/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/vscode/.local/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/vscode/.local/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


In [4]:
ruta = "../datos/procesados/dataset_modelo_tech.csv"
df_modelo.to_csv(ruta, index=False)

tamano_mb = Path(ruta).stat().st_size / (1024**2)
print(f"Guardado: dataset_modelo_tech.csv")
print(f"Tamaño: {tamano_mb:.1f} MB")
print(f"\n{'='*50}")
print("RESUMEN DEL DATASET TECNOLÓGICO")
print(f"{'='*50}")
print(f"  Observaciones:        {len(df_modelo):,}")
print(f"  Empresas:             {df_modelo['ticker'].nunique():,}")
print(f"  Features:             {len(features)}")
print(f"  Período:              {df_modelo['end'].min().date()} → {df_modelo['end'].max().date()}")
print(f"  Y=1 (breakeven):      {df_modelo['Y'].mean()*100:.1f}%")
print(f"  Y=0 (no breakeven):   {(1-df_modelo['Y'].mean())*100:.1f}%")

Guardado: dataset_modelo_tech.csv
Tamaño: 1.4 MB

RESUMEN DEL DATASET TECNOLÓGICO
  Observaciones:        3,390
  Empresas:             255
  Features:             49
  Período:              2008-10-03 → 2025-06-30
  Y=1 (breakeven):      56.7%
  Y=0 (no breakeven):   43.3%


HITO CUADERNO 2 — Construcción de features y variable objetivo
==================================================================
Fecha: 10/08/2026

Notebook: 02_construccion_features_tech.ipynb

Qué se hizo:
- Se aplicó el mismo protocolo metodológico del modelo general: limpieza
  de fechas, separación de variables de balance vs. flujo, destrimestra-
  lización de acumulados, y transformación a formato panel (pivot).
- Se construyó la variable objetivo Y (breakeven promedio a 4 trimestres),
  filtrando el universo a observaciones con FCF operativo negativo.
- Se calcularon 11 variables derivadas (burn rate, runway, márgenes,
  ratios de liquidez y apalancamiento, rotación de activos, intensidad
  de capex), con winsorización al percentil 1-99.
- Se recalculó el criterio de cobertura mínima (15%) específicamente
  sobre el universo tecnológico, identificando variables distintas a
  las eliminadas en el modelo general.

Resultado:
- Dataset final: 3.390 observaciones, 255 empresas, 49 variables
  predictoras (frente a 65 en el modelo general, por la eliminación de
  las 16 variables temporales que se agregarán más adelante si aplica)
- Balance de clases: 56.7% Y=1, 43.3% Y=0 (saludable, sin necesidad de
  corrección)
- Variables eliminadas por baja cobertura: deuda_corto_plazo (0.3%,
  consistente con el modelo general) e intereses_pagados (13.8%,
  distinta a la eliminada en el modelo general -- confirma que la
  cobertura de datos varía según el universo sectorial analizado)
- Archivo generado: datos/procesados/dataset_modelo_tech.csv

Importancia:
Este es el dataset definitivo sobre el cual se entrenará el modelo
predictivo (Cuaderno 3) bajo el protocolo walk-forward ajustado
(2018-2024, 7 folds), respondiendo directamente a la recomendación
del asesor de desarrollar un modelo sectorialmente específico.